In [1]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from matplotlib.patches import Patch
import json
from collections import Counter

from dataset import DFC18

In [ ]:
# load datasets
ds = DFC18(split="train")

Loaded 2888 samples from val set


In [3]:
img = ds.__getitem__(1)['image']
print(img.shape)
img = ds.__getitem__(1)['label']
print(img.shape)

(3, 256, 256)
(256, 256)


**Mean & Std**

In [4]:
sum_img = np.zeros((3,))  # Running sum for each band
sum_sq_img = np.zeros((3,))  # Running sum of squares for each band
num_pixels = 0  # Total number of pixels processed


for i in range(len(ds)):
    img = ds.__getitem__(i)['image']
    img = img.astype(np.float64)

    sum_img += np.sum(img, axis=(1, 2))  # Sum over height (axis 1) and width (axis 2)
    sum_sq_img += np.sum(img**2, axis=(1, 2))  # Sum of squared values for each band

    num_pixels += img.shape[1] * img.shape[2]  # 256 * 256

mean = sum_img / num_pixels  # Mean for each band
std = np.sqrt((sum_sq_img / num_pixels) - mean**2)  # Standard deviation for each band

print("Mean:", mean)
print("Std:", std)

# Save mean and std for later use
np.save("utilities/rgb_train_mean.npy", mean)
np.save("utilities/rgb_train_std.npy", std)

Mean: [118.6585854  121.60668922 120.02433118]
Std: [60.16847375 58.5987261  58.77801756]


**Majority Classes**

In [ ]:
majority_classes = []

for i in range(len(ds)):
    mask = ds.__getitem__(i)['label']
    mask = mask[mask != 0]
    class_counts = Counter(mask.flatten())
    majority_class = class_counts.most_common(1)[0][0]  # Get class with most pixels
    majority_classes.append(int(majority_class))

with open("utilities/train_majority_class_per_image.json", "w") as f:
    json.dump(majority_classes, f)

**Train Class Frequencies**

In [6]:
def compute_class_pixel_frequencies(label_dir, max_class=20):
    label_files = [f for f in os.listdir(label_dir) if f.endswith('_label.npy')]
    class_counts = np.zeros(max_class + 1, dtype=np.int64)  # includes class 0

    for fname in label_files:
        label_path = os.path.join(label_dir, fname)
        label = np.load(label_path)
        for c in range(max_class + 1):
            class_counts[c] += np.sum(label == c)

    # Exclude class 0
    class_counts_excl0 = class_counts[1:]
    total_pixels_excl0 = class_counts_excl0.sum()

    # Compute relative frequencies
    class_frequencies = class_counts_excl0 / total_pixels_excl0

    return class_frequencies  # This will sum to 1

# Example usage
train_label_dir = 'data/patches/train'
frequencies = compute_class_pixel_frequencies(train_label_dir)
print(frequencies)

[1.90555569e-02 6.57992732e-02 1.36151029e-03 2.67724482e-02
 9.97414111e-03 8.04829999e-03 4.61384373e-04 7.72459578e-02
 4.54671975e-01 8.62569455e-02 5.70884499e-02 3.10250849e-03
 9.54861768e-02 2.00251306e-02 1.39215162e-02 2.25124844e-02
 2.54609286e-04 1.32041993e-02 1.08724781e-02 1.38849544e-02]
